<a href='https://www.darshan.ac.in/'> <img src='https://www.darshan.ac.in/Content/media/DU_Logo.svg' width="250" height="300"/></a>
<pre>
<center><b><h1>Machine Learning - 2301CS621</b></center>

<center><b><h1>Project : Cardiovascular Disease Dataset</b></center>
<center><b><h3>Week 5: Advanced Model Training, Cross-Validation & Hyperparameter Tuning</h3></b></center>    
<pre>

### Step 1. Import Necessary Libraries

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### Step 2. Load Preprocessed & Scaled Dataset

In [2]:
df = pd.read_csv('cardio_cleaned_scaled.csv')
X = df.drop(columns=['cardio'])
y = df['cardio']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

Training shape: (52361, 12), Testing shape: (13091, 12)


### Step 3. Experiment with Multiple Models & Compare Validation Metrics
We train and evaluate 5 distinct algorithms across Accuracy, Precision, Recall, and F1-score:
1. **Logistic Regression** (Linear baseline)
2. **Decision Tree** (Non-linear baseline)
3. **Random Forest** (Bagging ensemble)
4. **AdaBoost** (Adaptive boosting ensemble)
5. **Gradient Boosting** (Sequential gradient descent boosting)

In [3]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
}

results = []
for name, m in models.items():
    m.fit(X_train, y_train)
    y_pred = m.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        'Model': name,
        'Accuracy': f"{acc*100:.2f}%",
        'Precision': f"{prec*100:.2f}%",
        'Recall': f"{rec*100:.2f}%",
        'F1-score': f"{f1*100:.2f}%"
    })

comparison_df = pd.DataFrame(results)
comparison_df

,Model,Accuracy,Precision,Recall,F1-score
0,Logistic Regression,72.65%,75.09%,68.56%,71.68%
1,Decision Tree,72.34%,73.50%,70.68%,72.06%
2,Random Forest (Bagging),73.03%,76.19%,67.72%,71.70%
3,AdaBoost,72.42%,76.69%,65.14%,70.45%
4,Gradient Boosting,72.97%,74.95%,69.74%,72.25%


### Step 4. K-Fold Cross-Validation for Model Stability
To ensure model stability and confirm the model does not suffer from data split bias, we perform **5-Fold Stratified Cross-Validation** on the Random Forest model.

In [4]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)

for fold, score in enumerate(cv_scores, 1):
    print(f"Fold {fold} Accuracy: {score*100:.2f}%")

print("-" * 35)
print(f"Mean CV Accuracy: {cv_scores.mean()*100:.2f}%")
print(f"Standard Deviation: {cv_scores.std()*100:.2f}% (High stability, low variance)")

Fold 1 Accuracy: 72.66%
Fold 2 Accuracy: 73.54%
Fold 3 Accuracy: 73.01%
Fold 4 Accuracy: 73.38%
Fold 5 Accuracy: 72.95%
-----------------------------------
Mean CV Accuracy: 73.11%
Standard Deviation: 0.31% (High stability, low variance)


### Step 5. Hyperparameter Tuning using GridSearchCV
We optimize key Random Forest hyperparameters (`max_depth`, `n_estimators`) using Grid Search with 3-fold cross-validation.

In [5]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [5, 10, 15]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("Best Hyperparameters:", grid_search.best_params_)
print(f"Best Cross-Validation Score: {grid_search.best_score_*100:.2f}%")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best Hyperparameters: {'max_depth': 10, 'n_estimators': 100}
Best Cross-Validation Score: 72.99%


### Step 6. Final Model Validation on Held-Out Test Set
We evaluate the tuned model on the unseen test set to confirm generalization.

In [6]:
best_model = grid_search.best_estimator_
y_final_pred = best_model.predict(X_test)

final_acc = accuracy_score(y_test, y_final_pred)
final_prec = precision_score(y_test, y_final_pred)
final_rec = recall_score(y_test, y_final_pred)
final_f1 = f1_score(y_test, y_final_pred)

print(f"Final Test Accuracy : {final_acc*100:.2f}%")
print(f"Final Test Precision: {final_prec*100:.2f}%")
print(f"Final Test Recall   : {final_rec*100:.2f}%")
print(f"Final Test F1-Score : {final_f1*100:.2f}%")

Final Test Accuracy : 73.03%
Final Test Precision: 76.19%
Final Test Recall   : 67.72%
Final Test F1-Score : 71.70%
